# dim_nationality.py

## Purpose
Builds `dim_nationality` by joining EA FC nationalities with ISO 3166-1 codes.

## Sources
- Bronze: `fifa.bronze.ea_fc_players_raw` (column `nationality`)
- Reference CSV: `/Volumes/fifa/default/fifa_landing/paises.csv`
  - Acquired from [this repository](https://gist.githubusercontent.com/brenes/1095110/raw/c8f208b03485ba28f97c500ab7271e8bce43b9c6/paises.csv)
  - Some countries were manually added and corrected (e.g. England, Wales, Scotland, Kosovo)
  -   - You must upload the CSV to the `fifa_landing` folder before running the pipeline or this cell.

## Strategy
- SCD1 (nationalities are stable)
- MERGE instead of overwrite (the catalog has ~164 rows, MERGE avoids rewriting all of them on every run)
- Name normalization (lowercase, no accents, trim) on both sides
- Left join by the `ea_name` column in the reference CSV
- Surrogate key: `xxhash64(iso3)` — deterministic across runs
- Manual mapping for EA-specific names (`England`, `Holland`, `Korea Republic`, ...)

## Output
- Table: `fifa.silver.dim_nationality`

## Columns
- `nationality_id` (INT): hash of `iso3`
- `nationality_code` (STRING): ISO3 code
- `nationality_name` (STRING): EA name
- `_ingestion_timestamp` (TIMESTAMP)

## Notes
- The `nationality_id` is a hash of the ISO3 code, which guarantees stable IDs across pipeline runs.
- Unmapped nationalities (those without a matching ISO3 in the reference CSV) are filtered out and not inserted into the dimension.

In [0]:
# --- Load and clean reference CSV ---
from pyspark.sql.functions import col, lower, translate, trim, current_timestamp, xxhash64, pmod
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

# Load the reference CSV (country names + ISO3 codes)
iso_ref = spark.read.csv(
    "/Volumes/fifa/default/fifa_landing/paises.csv",
    header=True,
    inferSchema=True
)

# Strip whitespace from column names
for c in iso_ref.columns:
    iso_ref = iso_ref.withColumnRenamed(c, c.strip())

# Select and clean the columns we need
iso_ref_clean = (
    iso_ref
    .select(
        # ea_name is a column added manually to match EA FC naming
        trim(col("ea_name")).alias("country_name_en"),
        trim(col("iso3")).alias("iso3")
    )
    .filter(col("iso3").isNotNull())
    .filter(col("iso3") != "")
)

# --- Normalize Bronze ---
# Helper to remove accents and lowercase
def normalize(c):
    return lower(translate(trim(c), "áéíóúÁÉÍÓÚñÑüÜ", "aeiouAEIOUnNuU"))

bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

bronze_nat = (
    bronze
    .select(col("nationality").alias("nationality_name"))
    .distinct()
    .withColumn("nat_norm", normalize(col("nationality_name")))
)

# --- Join CSV and Bronze ---
iso_ref_clean = iso_ref_clean.withColumn(
    "en_norm", normalize(col("country_name_en"))
)

joined = (
    bronze_nat
    .join(iso_ref_clean, bronze_nat.nat_norm == iso_ref_clean.en_norm, how="left")
)

# --- Build final dimension ---
mapped = joined.filter(col("iso3").isNotNull()) # keep only mapped nationalities
unmapped = joined.filter(col("iso3").isNull()) # keep unmapped for logging

dim_nationality = (
    mapped
    .withColumn("nationality_id", pmod(xxhash64(col("iso3")), 1000000).cast(IntegerType()))
    .withColumn("nationality_code", col("iso3"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select("nationality_id", "nationality_code", "nationality_name", "_ingestion_timestamp")
    .orderBy("nationality_id")
)

# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_nationality (
    nationality_id INT,
    nationality_code STRING,
    nationality_name STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# --- MERGE: insert new nationalities, update existing ones if the name changed ---
target = DeltaTable.forName(spark, "fifa.silver.dim_nationality")

(
    target.alias("t")
    .merge(
        dim_nationality.alias("s"),
        "t.nationality_code = s.nationality_code"
    )
    .whenMatchedUpdate(
        condition="t.nationality_name <> s.nationality_name",
        set={
            "nationality_name": "s.nationality_name",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .whenNotMatchedInsert(
        values={
            "nationality_id": "s.nationality_id",
            "nationality_code": "s.nationality_code",
            "nationality_name": "s.nationality_name",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .execute()
)

print(f"dim_nationality creada con {spark.read.table('fifa.silver.dim_nationality').count()} filas")

LIMIT 10 of the dim_nationality table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_nationality ORDER BY nationality_id LIMIT 10;